# Kaggle MoneyPrinterTurbo + IndicF5

This notebook starts IndicF5 API (`8765`) and your MoneyPrinterTurbo Streamlit UI (`8501`) as separate processes, then automatically creates a Cloudflare Quick Tunnel and prints the public web URL. It also exposes direct Manual Timeline JSON generation. Run every cell in order. Enable GPU and Internet in Kaggle settings.

**Before first run:** add Kaggle Secrets `HF_TOKEN` (after accepting the `ai4bharat/IndicF5` model access conditions). Add `GITHUB_TOKEN` only if your IndicF5 repository is private. The notebook uses MoneyPrinterTurbo branch `feature/kaggle-indicf5-pipeline` and IndicF5 branch `main`. Cloudflare tunneling is enabled by default; the URL is public to anyone who has it, so do not share it if the UI contains private content. Never paste tokens into notebook cells.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time, base64, shutil, textwrap
import requests

# ---- Project settings (safe to edit; never put tokens here) ----
MPT_REPO = "https://github.com/rjosephm/MoneyPrinterTurbo.git"
MPT_BRANCH = "feature/kaggle-indicf5-pipeline"
INDICF5_PRIVATE_REPO = "rjosephm/indicF5"
INDICF5_BRANCH = "main"  # Use the main branch of your IndicF5 repository
MODEL_DATASET_DIR = Path("/kaggle/input/indicf5-model")  # Optional Kaggle Dataset containing a HF model snapshot
INDICF5_REFERENCE_AUDIO = ""  # Auto-selects maleSample/M01.wav if left blank
INDICF5_REFERENCE_TEXT = ""   # Auto-reads the exact transcript from maleSample/mapping.csv if blank
ENABLE_PUBLIC_TUNNEL = True  # Automatically publish the Streamlit UI using a temporary public Cloudflare Quick Tunnel

WORK = Path('/kaggle/working')
MPT_ROOT = WORK / 'MoneyPrinterTurbo'
PRIVATE_INDICF5_ROOT = WORK / 'IndicF5Private'
OUT_DIR = WORK / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE = WORK / 'hf-cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['INDICF5_API_URL'] = 'http://127.0.0.1:8765'
os.environ['INDICF5_REFERENCE_AUDIO'] = INDICF5_REFERENCE_AUDIO
os.environ['INDICF5_REFERENCE_TEXT'] = INDICF5_REFERENCE_TEXT

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    def get_secret(name):
        try: return _secrets.get_secret(name)
        except Exception: return ''
except Exception:
    def get_secret(name): return ''

HF_TOKEN = get_secret('HF_TOKEN')
GITHUB_TOKEN = get_secret('GITHUB_TOKEN')
PEXELS_API_KEY = get_secret('PEXELS_API_KEY')
if PEXELS_API_KEY:
    os.environ['PEXELS_API_KEY'] = PEXELS_API_KEY
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
else:
    print('WARNING: Kaggle Secret HF_TOKEN is missing. Gated Hugging Face model download may fail.')

print('Workspace:', WORK)
print('GPU check:')
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA GPU')"], check=False)


In [ ]:
# Clone the selected MoneyPrinterTurbo feature branch.
if MPT_ROOT.exists():
    shutil.rmtree(MPT_ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', MPT_BRANCH, MPT_REPO, str(MPT_ROOT)], check=True)

# Clone the IndicF5 project and its reference samples. A token is optional while the repo is public;
# once you change the repository to private, add GITHUB_TOKEN in Kaggle Secrets.
if PRIVATE_INDICF5_ROOT.exists(): shutil.rmtree(PRIVATE_INDICF5_ROOT)
repo_slug = INDICF5_PRIVATE_REPO.strip().removesuffix('.git')
repo_url = 'https://github.com/' + repo_slug + '.git'
if GITHUB_TOKEN:
    auth = base64.b64encode(('x-access-token:' + GITHUB_TOKEN).encode()).decode()
    git_env = os.environ.copy()
    git_env['GIT_CONFIG_COUNT'] = '1'
    git_env['GIT_CONFIG_KEY_0'] = 'http.extraheader'
    git_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + auth
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', INDICF5_BRANCH, repo_url, str(PRIVATE_INDICF5_ROOT)], env=git_env, check=True, capture_output=True, text=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', INDICF5_BRANCH, repo_url, str(PRIVATE_INDICF5_ROOT)], check=True)
print('IndicF5 source and voice samples cloned:', INDICF5_PRIVATE_REPO, 'branch', INDICF5_BRANCH)

# Auto-select a real WAV and its exact transcript from the repository's mapping CSV.
import csv
if not INDICF5_REFERENCE_AUDIO or not INDICF5_REFERENCE_TEXT.strip():
    mapping = PRIVATE_INDICF5_ROOT / 'maleSample' / 'mapping.csv'
    if not mapping.is_file():
        raise FileNotFoundError(f'Voice mapping CSV not found: {mapping}')
    with mapping.open('r', encoding='utf-8-sig', newline='') as fh:
        rows = list(csv.DictReader(fh))
    chosen = next((r for r in rows if (r.get('filename') or '').strip() == 'M01.wav'), None)
    if not chosen:
        raise RuntimeError('Could not find M01.wav in maleSample/mapping.csv')
    INDICF5_REFERENCE_AUDIO = str((PRIVATE_INDICF5_ROOT / 'maleSample' / chosen['filename']).resolve())
    INDICF5_REFERENCE_TEXT = (chosen.get('text') or '').strip()
    if not INDICF5_REFERENCE_TEXT:
        raise RuntimeError('M01.wav transcript is empty in mapping.csv')
    os.environ['INDICF5_REFERENCE_AUDIO'] = INDICF5_REFERENCE_AUDIO
    os.environ['INDICF5_REFERENCE_TEXT'] = INDICF5_REFERENCE_TEXT
print('Default IndicF5 reference:', Path(INDICF5_REFERENCE_AUDIO).name)

# IMPORTANT: rjosephm/indicF5 is the user's voice-sample/profile repository,
# not the model source package; it intentionally has no setup.py or f5_tts/.
# The Hugging Face model's remote code imports f5_tts, provided by the official
# AI4Bharat/IndicF5 source. Install code WITHOUT deps so pip does not silently
# replace Kaggle's pre-pinned Torch/NumPy/Transformers stack.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-deps',
    'git+https://github.com/AI4Bharat/IndicF5.git'
], check=True)
subprocess.run([
    sys.executable, '-c',
    'import f5_tts; print("Official f5_tts package:", f5_tts.__file__)'
], check=True)

# torchdiffeq is an official f5_tts runtime dependency. Install only this pure-Python
# package without dependency resolution so Kaggle's existing Torch/SciPy stack is preserved.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-deps', 'torchdiffeq==0.2.5'
], check=True)
subprocess.run([
    sys.executable, '-c',
    'import torchdiffeq; print("torchdiffeq import OK:", getattr(torchdiffeq, "__version__", "version unknown"))'
], check=True)

# Fix the IndicF5/F5-TTS runtime dependency chain without changing Kaggle's
# existing PyTorch, CUDA, or NumPy versions. x-transformers 2.x introduced
# einx/torch_einops_utils imports; F5-TTS supports the compatible 1.31.14 line.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-deps',
    'x-transformers==1.31.14',
    'cached-path',
    'ema-pytorch>=0.5.2',
    'hydra-core>=1.3.0',
    'pypinyin',
    'tomli',
    'transformers_stream_generator',
], check=True)
subprocess.run([
    sys.executable, '-c',
    'import torchdiffeq, x_transformers, cached_path, ema_pytorch, hydra, pypinyin, transformers_stream_generator; print("IndicF5 runtime imports OK")'
], check=True)

# Install the project's pinned dependencies and the known-compatible IndicF5 inference stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(MPT_ROOT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==4.49.0', 'huggingface_hub==0.36.2', 'gradio==5.50.0', 'diffusers==0.35.2', 'soundfile', 'safetensors', 'vocos', 'accelerate', 'sentencepiece'], check=True)

# Verify the selected Hub/Transformers/UI dependency versions after installation.
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=False)

# Prefer a model snapshot attached as a Kaggle Dataset if one exists; otherwise use Hugging Face.
local_model = None
if MODEL_DATASET_DIR.exists():
    for cfg in MODEL_DATASET_DIR.rglob('config.json'):
        if list(cfg.parent.glob('*.safetensors')):
            local_model = cfg.parent
            break
if local_model:
    os.environ['INDICF5_MODEL_ID'] = str(local_model)
    print('Using model files from Kaggle Dataset:', local_model)
else:
    os.environ['INDICF5_MODEL_ID'] = 'ai4bharat/IndicF5'
    print('Using Hugging Face model id: ai4bharat/IndicF5')

# Use the Kaggle-adapted API adapter, not the repo's current Windows-specific API file.
# This adapter reads reference paths from the request and puts the model on the selected GPU.
INDICF5_API_SCRIPT = MPT_ROOT / 'kaggle' / 'indicf5_api.py'
print('IndicF5 API adapter:', INDICF5_API_SCRIPT)
if not (MPT_ROOT / 'kaggle' / 'mpt_kaggle.py').is_file(): raise FileNotFoundError('Kaggle MPT launcher is missing from the selected branch.')


In [ ]:
# Start IndicF5 in its own Python process, then wait for the health endpoint.
API_LOG = open(WORK / 'indicf5_api.log', 'w', encoding='utf-8')
api_env = os.environ.copy()
api_env['INDICF5_HOST'] = '127.0.0.1'
api_env['INDICF5_PORT'] = '8765'
INDICF5_PROCESS = subprocess.Popen([sys.executable, str(INDICF5_API_SCRIPT)], cwd=str(INDICF5_API_SCRIPT.parent), env=api_env, stdout=API_LOG, stderr=subprocess.STDOUT)

health = None
deadline = time.time() + 900  # model download/loading can take several minutes
while time.time() < deadline:
    if INDICF5_PROCESS.poll() is not None:
        break
    try:
        response = requests.get('http://127.0.0.1:8765/health', timeout=3)
        if response.ok:
            health = response.json()
            if health.get('ok'):
                break
            if health.get('error'):
                print('IndicF5 reports a model-load error:', health['error'])
                break
    except requests.RequestException:
        pass
    time.sleep(5)
print('IndicF5 health:', health)
if not health or not health.get('ok'):
    API_LOG.flush()
    print('Last health payload:', health)
    print((WORK / 'indicf5_api.log').read_text(encoding='utf-8', errors='replace')[-12000:])
    raise RuntimeError('IndicF5 model failed to load. The health payload and API log above show the specific cause; fix that before continuing.')
if not health.get('cuda_available'):
    raise RuntimeError('IndicF5 loaded without CUDA. Check Kaggle Settings → Accelerator → GPU before rendering.')

# Mandatory Malayalam smoke test using the reference WAV/transcript loaded from your repo.
if not INDICF5_REFERENCE_AUDIO or not Path(INDICF5_REFERENCE_AUDIO).is_file() or not INDICF5_REFERENCE_TEXT.strip():
    raise RuntimeError('Reference WAV or transcript is missing. Check the cloned repo and mapping CSV.')
test_response = requests.post('http://127.0.0.1:8765/generate', json={
    'text': 'നമസ്കാരം. ഇത് ഇൻഡിക് എഫ് ഫൈവ് ശബ്ദ പരിശോധനയാണ്.',
    'reference_audio': INDICF5_REFERENCE_AUDIO,
    'reference_text': INDICF5_REFERENCE_TEXT,
}, timeout=(10, 900))
test_response.raise_for_status()
if not test_response.content.startswith(b'RIFF') or test_response.content[8:12] != b'WAVE':
    raise RuntimeError('IndicF5 smoke test did not return a WAV file.')
SMOKE_WAV = WORK / 'indicf5_smoke_test.wav'
SMOKE_WAV.write_bytes(test_response.content)
if SMOKE_WAV.stat().st_size < 1000: raise RuntimeError('IndicF5 smoke-test WAV is unexpectedly small.')
print('PASS: IndicF5 health and Malayalam WAV test passed:', SMOKE_WAV, 'bytes:', SMOKE_WAV.stat().st_size)


In [ ]:
# Generate MoneyPrinterTurbo voice profiles from both sample folders and their mapping.csv files.
# This makes the repository's male/female samples selectable in Manual Timeline.
profile_dir = MPT_ROOT / 'storage' / 'indicf5_profiles'
profile_dir.mkdir(parents=True, exist_ok=True)
created_profiles = 0
for folder_name, prefix in [('maleSample', 'ml-male'), ('femaleSample', 'ml-female')]:
    sample_dir = PRIVATE_INDICF5_ROOT / folder_name
    mapping = sample_dir / 'mapping.csv'
    if not mapping.is_file():
        print('Skipping profile folder without mapping.csv:', folder_name)
        continue
    with mapping.open('r', encoding='utf-8-sig', newline='') as fh:
        rows = list(csv.DictReader(fh))
    for row in rows:
        filename = (row.get('filename') or '').strip()
        transcript = (row.get('text') or '').strip()
        wav = (sample_dir / filename).resolve()
        if not filename or not transcript or not wav.is_file() or wav.suffix.lower() != '.wav':
            continue
        profile_id = prefix + '-' + Path(filename).stem.lower().replace('_', '-')
        profile = {'id': profile_id, 'name': profile_id, 'reference_audio': str(wav),
                   'reference_text': transcript, 'api_url': 'http://127.0.0.1:8765'}
        (profile_dir / f'{profile_id}.json').write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding='utf-8')
        created_profiles += 1
default_profile = {'id': 'kaggle-default', 'name': 'Kaggle IndicF5 (M01)',
                   'reference_audio': str(Path(INDICF5_REFERENCE_AUDIO).resolve()),
                   'reference_text': INDICF5_REFERENCE_TEXT, 'api_url': 'http://127.0.0.1:8765'}
(profile_dir / 'kaggle-default.json').write_text(json.dumps(default_profile, ensure_ascii=False, indent=2), encoding='utf-8')
print('IndicF5 voice profiles prepared:', created_profiles + 1, '(including kaggle-default)')

# Start MoneyPrinterTurbo in a separate Python process. The IndicF5 process remains running independently.
MPT_LOG = open(WORK / 'moneyprinterturbo_streamlit.log', 'w', encoding='utf-8')
mpt_env = os.environ.copy()
mpt_env['INDICF5_API_URL'] = 'http://127.0.0.1:8765'
mpt_env['INDICF5_REFERENCE_DIR'] = str(Path(INDICF5_REFERENCE_AUDIO).resolve().parent)
MPT_PROCESS = subprocess.Popen([sys.executable, str(MPT_ROOT / 'kaggle' / 'mpt_kaggle.py')], cwd=str(MPT_ROOT), env=mpt_env, stdout=MPT_LOG, stderr=subprocess.STDOUT)
time.sleep(8)
if MPT_PROCESS.poll() is not None:
    MPT_LOG.flush()
    print((WORK / 'moneyprinterturbo_streamlit.log').read_text(encoding='utf-8', errors='replace')[-8000:])
    raise RuntimeError('MoneyPrinterTurbo UI process exited during startup.')
print('MoneyPrinterTurbo UI process started on port 8501. Process ID:', MPT_PROCESS.pid)
print('IndicF5 process ID:', INDICF5_PROCESS.pid)

def generate_from_json(json_path, cleanup=False, output_dir=None):
    """Run Manual Timeline JSON rendering directly without opening Streamlit."""
    json_path = Path(json_path).expanduser().resolve()
    if not json_path.is_file(): raise FileNotFoundError(json_path)
    if not requests.get('http://127.0.0.1:8765/health', timeout=5).json().get('ok'):
        raise RuntimeError('IndicF5 API is not healthy; refusing to start full render.')
    cmd = [sys.executable, str(MPT_ROOT / 'kaggle' / 'generate_from_json.py'), str(json_path)]
    if cleanup: cmd.append('--cleanup')
    if output_dir: cmd.extend(['--output-dir', str(Path(output_dir).resolve())])
    env = os.environ.copy()
    env['INDICF5_API_URL'] = 'http://127.0.0.1:8765'
    env['INDICF5_REFERENCE_AUDIO'] = INDICF5_REFERENCE_AUDIO
    env['INDICF5_REFERENCE_TEXT'] = INDICF5_REFERENCE_TEXT
    result = subprocess.run(cmd, cwd=str(MPT_ROOT), env=env, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr: print(result.stderr[-6000:])
    if result.returncode != 0: raise RuntimeError(f'JSON video generation failed with exit code {result.returncode}')
    final_line = next((line for line in result.stdout.splitlines() if line.startswith('FINAL_VIDEO=')), None)
    if not final_line: raise RuntimeError('Generation finished without reporting FINAL_VIDEO path.')
    rendered = Path(final_line.split('=', 1)[1].strip())
    if not rendered.is_file(): raise FileNotFoundError(f'Final video was not found: {rendered}')
    saved = OUT_DIR / rendered.name
    shutil.copy2(rendered, saved)
    print('FINAL_VIDEO_COPY=', saved)
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(saved)))
    except Exception:
        pass

print('Ready. Example: generate_from_json("/kaggle/working/my_video.json", cleanup=False)')


In [ ]:
# Automatically expose the Streamlit UI through a Cloudflare Quick Tunnel.
# WARNING: this creates a public URL. Anyone who obtains it may reach the UI; do not expose sensitive projects or credentials.
if ENABLE_PUBLIC_TUNNEL:
    cloudflared = WORK / 'cloudflared'
    subprocess.run(['bash', '-lc', f'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cloudflared} && chmod +x {cloudflared}'], check=True)
    TUNNEL_LOG = open(WORK / 'cloudflared.log', 'w', encoding='utf-8')
    TUNNEL_PROCESS = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT)
    import re
    public_url = None
    for _ in range(60):
        time.sleep(2); TUNNEL_LOG.flush()
        logs = (WORK / 'cloudflared.log').read_text(encoding='utf-8', errors='replace')
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
        if match: public_url = match.group(0); break
    print('Streamlit public URL:', public_url or 'Tunnel URL not found; inspect cloudflared.log')
    if public_url:
        try:
            from IPython.display import Markdown, display
            display(Markdown(f'### [Open MoneyPrinterTurbo in your browser]({public_url})'))
        except Exception:
            pass
else:
    print('Cloudflare tunnel is disabled. Set ENABLE_PUBLIC_TUNNEL=True in the configuration cell and rerun this cell to expose the UI.')


## Generate a video from JSON

Upload your Manual Timeline JSON into `/kaggle/working` (or attach a dataset containing it), then run a code cell such as:

```python
generate_from_json('/kaggle/working/my_video.json', cleanup=False)
```

Use `cleanup=True` only after you are happy with the result. Final videos are saved under `storage/manual_timeline/<project_id>/` by default. Copy final MP4 files to `/kaggle/working/outputs` if you want them easy to download. Kaggle sessions are temporary, so download outputs or save them to persistent storage.
